### Qualitative Analysis: Fine-tuned BERT

In [1]:
# Import necessary libraries
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np

In [5]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify the model path exists
import os
model_path = '/content/drive/MyDrive/bert_sentiment_model/final'
print(f"Model path exists: {os.path.exists(model_path)}")

Mounted at /content/drive
Model path exists: True


In [6]:
# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
# Create sentiment label mapping
sentiment_labels = {0: 'negative', 1: 'neutral', 2: 'positive'}

# Load examples from test set
test_data = pd.read_csv('sentiment-topic-test.tsv', sep='\t')
test_examples = test_data['sentence'].tolist()
true_labels = test_data['sentiment'].tolist()

In [13]:
def predict_sentiment(text):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Move to same device as model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted class
    predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

    # Convert to sentiment labels
    return [sentiment_labels[pred] for pred in predictions]

In [15]:
# Run predictions on all test examples and show only incorrect predictions
print("\nIncorrectly Predicted Examples:")
incorrect_count = 0

for i, example in enumerate(test_examples):
    predicted = predict_sentiment(example)[0]
    if predicted != true_labels[i]:
        incorrect_count += 1
        print(f"Example #{incorrect_count}:")
        print(f"Text: {example}")
        print(f"True label: {true_labels[i]}")
        print(f"Predicted: {predicted}")
        print("="*150)


Incorrectly Predicted Examples:
Example #1:
Text: It's still 0-0 so far, so way too early to tell - both teams trying their hardest, but maybe it won't be enough?
True label: neutral
Predicted: negative
Example #2:
Text: Did you hear the screenplay for it was originally written on a napkin?
True label: neutral
Predicted: negative
Example #3:
Text: It's really incredibly impressive to mess up such a tested blockbuster formula.
True label: negative
Predicted: positive
